In [1]:
import random

known_weather_data = {
    'berlin': 20.0
}

def get_weather(city: str) -> float:
    city = city.strip().lower()

    if city in known_weather_data:
        return known_weather_data[city]

    return round(random.uniform(-5, 35), 1)

In [3]:
# Describe what the function look like for our agent 

get_weather_tool = {
    "type": "function",
    "name": "get_weather", # Get weather 
    "description": "Get the current weather for a given city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The name of the specific city to get the weather for."
            }
        },
        "required": ["city"], # Array with required parameter names (in this case, the city name) 
        "additionalProperties": False
    }
}

- Q1 Answer: city

## Q2: Adding another tool

In [5]:
def set_weather(city: str, temp: float) -> None:
    city = city.strip().lower()
    known_weather_data[city] = temp
    return 'OK'

In [ ]:
# Q2 Answer: Write description for the set_weather function

set_weather_tool = {
    "type": "function",
    "name": "set_weather",
    "description": "Set the temperature for a specified city",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The name of the city to set weather for"
            },
            "temp": {
                "type": "number",  # float = "number" in JSON schema
                "description": "The temperature value to set"
            }
        },
        "required": ["city", "temp"],
        "additionalProperties": False
    }
}

# Q3- Model-Contect Protocol (MCP) 

- MCP allows LLMs communicate with different tools (like Qdrant). It's function calling, but one step further:
    - A tool can export a list of functions it has
    - When we include the tool to our Agent, we just need to include the link to the MCP server

In [8]:
# Install a library for MCP- FastMCP

!pip install fastmcp

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Preparing metadata (setup.py) ... done
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 10.9 MB/s eta 0:00:00
  DEPRECATION: Building 'pyperclip' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pyperclip'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for pyperclip: filename=pyperclip-1.9.0-py3-none-any.whl size=11003 sha256=d9d5b4a93dd0a9c73ddcdfff2d713ace93f95899bb318b78008f872562d87554
  Stored in directory: /Users/jenniferwang/Library/Caches/pip/wheels/e8/e7/56/591cb88ba1783b38c40d584026e766aac9c3a048e34128ce8b
Successfully built py

- Q3 Answer (Version of MCP): 2.10.5

## Q4: Create simple MCP Server

- Create a server for MCP-- see file 'weather_server.py'
  

In [13]:
!pip install nest_asyncio

In [19]:
import nest_asyncio
nest_asyncio.apply()

from fastmcp import FastMCP
import random

known_weather_data = {'berlin': 20.0}
mcp = FastMCP("Demo 🚀")

@mcp.tool
def get_weather(city: str) -> float:
    """Retrieves the temperature for a specified city."""
    city = city.strip().lower()
    if city in known_weather_data:
        return known_weather_data[city]
    return round(random.uniform(-5, 35), 1)

@mcp.tool
def set_weather(city: str, temp: float) -> None:
    """Sets the temperature for a specified city."""
    city = city.strip().lower()
    known_weather_data[city] = temp
    return 'OK'

# Wrap the async run method to avoid blocking the Jupyter notebook
async def main(): 
    await mcp.run_async()  
    # Alternative is to put transport="http" in the run_async() above # Use HTTP transport for Jupyter/VS Code
    
await main()

╭─ FastMCP 2.0 ──────────────────────────────────────────────────────────────╮
│                                                                            │
│        _ __ ___ ______           __  __  _____________    ____    ____     │
│       _ __ ___ / ____/___ ______/ /_/  |/  / ____/ __ \  |___ \  / __ \    │
│      _ __ ___ / /_  / __ `/ ___/ __/ /|_/ / /   / /_/ /  ___/ / / / / /    │
│     _ __ ___ / __/ / /_/ (__  ) /_/ /  / / /___/ ____/  /  __/_/ /_/ /     │
│    _ __ ___ /_/    \__,_/____/\__/_/  /_/\____/_/      /_____(_)____/      │
│                                                                            │
│                                                                            │
│                                                                            │
│    🖥️  Server name:     Demo 🚀                                             │
│    📦 Transport:       STDIO                                               │
│                                                                            │
│    📚 Docs:            https://gofastmcp.com                               │
│    🚀 Deploy:          https://fastmcp.cloud                               │
│                                                                            │
│    🏎️  FastMCP version: 2.10.5                                              │
│    🤝 MCP version:     1.11.0                                              │
│                                                                            │
╰────────────────────────────────────────────────────────────────────────────╯

AttributeError: 'OutStream' object has no attribute 'buffer'

- Q4 answer: stdio

## Q5: MCP communication

In [21]:
import nest_asyncio
nest_asyncio.apply()

from fastmcp import FastMCP, Client
import random
import asyncio

# Set up server
known_weather_data = {'berlin': 20.0}
mcp = FastMCP("Demo 🚀")

@mcp.tool
def get_weather(city: str) -> float:
    """Retrieves the temperature for a specified city."""
    city = city.strip().lower()
    if city in known_weather_data:
        return known_weather_data[city]
    return round(random.uniform(-5, 35), 1)

# Test the call
async def test_weather_call():
    async with Client(mcp) as client:
        # This is equivalent to the JSON-RPC call
        result = await client.call_tool("get_weather", {"city": "berlin"})
        print("Response:", result)
        return result

# Run the test
result = await test_weather_call()

# Double check
print("Raw data:", result.data)  # Should be 20.0
print("Text content:", result.content[0].text)  # Should be "20.0"
print("Is error:", result.is_error)  # Should be False

Response: CallToolResult(content=[TextContent(type='text', text='20.0', annotations=None, meta=None)], structured_content={'result': 20.0}, data=20.0, is_error=False)
Raw data: 20.0
Text content: 20.0
Is error: False


- Q5 Answer: 
  CallToolResult(content=[TextContent(type='text', text='20.0', annotations=None, meta=None)], structured_content={'result': 20.0}, data=20.0, is_error=False

## Q6: MCP Client

In [23]:
# Set up the MCP client (get_weather and set_weather)
import nest_asyncio
nest_asyncio.apply()

import json
from pprint import pprint

from fastmcp import FastMCP, Client
import random

# Set up the server (same as before)
known_weather_data = {'berlin': 20.0}
mcp = FastMCP("Demo 🚀")

@mcp.tool
def get_weather(city: str) -> float:
    """
    Retrieves the temperature for a specified city.

    Parameters:
        city (str): The name of the city for which to retrieve weather data.

    Returns:
        float: The temperature associated with the city.
    """
    city = city.strip().lower()
    if city in known_weather_data:
        return known_weather_data[city]
    return round(random.uniform(-5, 35), 1)

@mcp.tool
def set_weather(city: str, temp: float) -> None: # Set weather function 
    """
    Sets the temperature for a specified city.

    Parameters:
        city (str): The name of the city for which to set the weather data.
        temp (float): The temperature to associate with the city.

    Returns:
        str: A confirmation string 'OK' indicating successful update.
    """
    city = city.strip().lower()
    known_weather_data[city] = temp
    return 'OK'

# Q6: Create MCP client and get tools list
async def get_tools_pretty(): 
    async with Client(mcp) as client:
        tools = await client.list_tools()
        
        print("=== TOOLS LIST (JSON FORMAT) ===")
        
        # Convert to dict and pretty print as JSON-- Print out in pretty print 
        tools_dict = [tool.model_dump() for tool in tools]
        print(json.dumps(tools_dict, indent=2))
        
        return tools
# async def get_available_tools():
#     async with Client(mcp) as client:
#         # Get list of available tools
#         tools = await client.list_tools()
#         print("Available tools:")
#         print(tools)
#         return tools

# Run it
# tools_result = await get_available_tools()
tools_result = await get_tools_pretty()

=== TOOLS LIST (JSON FORMAT) ===
[
  {
    "name": "get_weather",
    "title": null,
    "description": "Retrieves the temperature for a specified city.\n\nParameters:\n    city (str): The name of the city for which to retrieve weather data.\n\nReturns:\n    float: The temperature associated with the city.",
    "inputSchema": {
      "properties": {
        "city": {
          "title": "City",
          "type": "string"
        }
      },
      "required": [
        "city"
      ],
      "type": "object"
    },
    "outputSchema": {
      "properties": {
        "result": {
          "title": "Result",
          "type": "number"
        }
      },
      "required": [
        "result"
      ],
      "title": "_WrappedResult",
      "type": "object",
      "x-fastmcp-wrap-result": true
    },
    "annotations": null,
    "meta": null
  },
  {
    "name": "set_weather",
    "title": null,
    "description": "Sets the temperature for a specified city.\n\nParameters:\n    city (str): The n

- Q6 Answer: 
  [
  {
    "name": "get_weather",
    "title": null,
    "description": "Retrieves the temperature for a specified city.\n\nParameters:\n    city (str): The name of the city for which to retrieve weather data.\n\nReturns:\n    float: The temperature associated with the city.",
    "inputSchema": {
      "properties": {
        "city": {
          "title": "City",
          "type": "string"
        }
      },
      "required": [
        "city"
      ],
      "type": "object"
    },
    "outputSchema": {
      "properties": {
        "result": {
          "title": "Result",
          "type": "number"
        }
      },
...
    "annotations": null,
    "meta": null
  }
]